# Lesson 7.4 — Cost Optimization: Cost Explorer, Savings Plans & Tagging

**Level: Advanced** | **Exam:** SAP-C02 Domain 1, Task 1.5; Domain 3, Task 3.5

## AWS Cost Optimization

### Pricing Model Selection Guide
| Workload | Recommended Model | Reason |
|---------|-----------------|--------|
| Unpredictable, spiky | On-Demand | Pay only when running |
| Steady-state servers | Compute Savings Plans | Up to 66% off, flexible |
| Specific EC2 family | EC2 Instance Savings Plans | Up to 72% off |
| Interruptible batch | Spot Instances | Up to 90% off |
| DR / standby | Reserved (1yr No Upfront) | Reserved for predictable standby |

### Tagging Strategy
Tags enable granular cost attribution:
```
Required Tags:
  Environment: dev | staging | prod
  Team:        platform | product | data | security
  Project:     projectname
  CostCenter:  CC-1001
  Owner:       team-email@company.com
```

**Activate tags** in AWS Billing Console → Cost Allocation Tags → Activate user-defined tags

### Cost Optimization Workflow
1. **Enable** Cost Explorer + activate cost allocation tags
2. **Analyze** usage with Cost Explorer + Trusted Advisor  
3. **Right-size** using Compute Optimizer recommendations
4. **Commit** with Savings Plans (after 2-3 months of baseline data)
5. **Monitor** with AWS Budgets alerts
6. **Automate**: Schedule stop/start for dev environments

### Common Cost Leaks to Find
- Unused EBS volumes (detached, not snapshot-only)
- Unattached Elastic IPs ($0.005/hr when not attached)
- Idle load balancers with no targets
- Old snapshots accumulating
- Forgotten NAT Gateways

In [ ]:
import boto3, json
account_id = boto3.client('sts').get_caller_identity()['Account']
print(f'Account: {account_id}')

In [ ]:
# Cost analysis
import boto3
from datetime import datetime, timedelta

ce = boto3.client('ce', region_name='us-east-1')

end = datetime.now().strftime('%Y-%m-%d')
start = (datetime.now() - timedelta(days=30)).strftime('%Y-%m-%d')

try:
    response = ce.get_cost_and_usage(
        TimePeriod={'Start': start, 'End': end},
        Granularity='MONTHLY',
        Metrics=['UnblendedCost'],
        GroupBy=[{'Type': 'DIMENSION', 'Key': 'SERVICE'}]
    )
    
    print(f"AWS Costs by Service (last 30 days):")
    all_costs = []
    for result in response['ResultsByTime']:
        for group in result['Groups']:
            cost = float(group['Metrics']['UnblendedCost']['Amount'])
            if cost > 0.01:
                all_costs.append((group['Keys'][0], cost))
    
    for service, cost in sorted(all_costs, key=lambda x: x[1], reverse=True):
        print(f"  {service:50} ${cost:.4f}")
    print(f"  {'TOTAL':50} ${sum(c for _,c in all_costs):.4f}")
    
except Exception as e:
    print(f"Cost Explorer: {e}")
    print("Note: Cost Explorer activation takes up to 24h to show data")

## Exercise

In [ ]:
# TODO: Write a cost optimization audit function that finds common cost leaks:
# 1. Unattached EBS volumes (state = "available")
# 2. Unassociated Elastic IPs
# 3. Load balancers with no registered targets
# For each finding, calculate approximate monthly cost

import boto3

def cost_leak_audit(region='us-east-1'):
    ec2 = boto3.client('ec2', region_name=region)
    elbv2 = boto3.client('elbv2', region_name=region)
    findings = []
    
    # TODO: Find unattached EBS volumes
    # ec2.describe_volumes(Filters=[{'Name': 'status', 'Values': ['available']}])
    
    # TODO: Find unassociated Elastic IPs
    # ec2.describe_addresses() - filter where AssociationId is missing
    
    # TODO: Find LBs with no targets
    # elbv2.describe_load_balancers() -> describe_target_groups() -> describe_target_health()
    
    return findings

results = cost_leak_audit()
if results:
    total_monthly_waste = sum(r.get('monthly_cost', 0) for r in results)
    print(f"Found {len(results)} cost optimization opportunities")
    print(f"Estimated monthly waste: ${total_monthly_waste:.2f}")
    for r in results:
        print(f"  {r.get('type')}: {r.get('resource_id')} - ${r.get('monthly_cost', 0):.2f}/mo")

## Summary
- **Savings Plans (Compute)**: Most flexible, up to 66% off — recommended over RIs for most workloads
- **Spot Instances**: 90% off but interruptible — use for batch, CI/CD, dev environments
- **Tagging**: Activate cost allocation tags to attribute costs to teams/projects
- **Compute Optimizer**: ML-based right-sizing for EC2, Lambda, ECS (needs 14+ days data)
- **Common leaks**: Unattached EBS, idle EIPs, unassociated Elastic IPs — run monthly audit

---
**Next:** [Phase 8 — New Solutions: HA, Performance & Serverless](../phase-8-new-solutions/lesson-8.1-ha-fault-tolerant.ipynb)